# Real-Time FRB vs RFI Classifier

This notebook develops a lightweight CNN classifier for real-time Fast Radio Burst (FRB) detection, optimized for FPGA deployment with sub-millisecond latency requirements.

## Architecture Goals:
- **Latency**: Sub-millisecond inference (<100 microseconds target)
- **Accuracy**: >95% to reliably trigger the reinforcement learning agent
- **FPGA-friendly**: Small model size, minimal dense layer neurons (<1000 flatten output)
- **Hardware**: Xilinx VU9P FPGA deployment via hls4ml

## 1. Import Dependencies

In [ ]:
import glob
import os
import h5py
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path-

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
from tensorflow.image import resize

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, 
    confusion_matrix, 
    classification_report,
    roc_auc_score, 
    roc_curve,
    precision_recall_curve,
    average_precision_score
)

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

TensorFlow version: 2.20.0
GPU Available: []


## 2. Configure Data Paths

**IMPORTANT**: Update these paths to match your local directory structure!

In [15]:
# Update these paths to match your directory structure
FRB_H5_DIR = "./realtime_data/FRBs/"  # Directory containing FRB*.h5 files
RFI_PKL_PATH = "./RFIs/LOFAR_Full_RFI_dataset.pkl"  # Path to RFI pickle file

# Image size for training (must match FPGA deployment requirements)
TARGET_SIZE = (64, 64)  # Optimized for FPGA synthesis
NUM_CLASSES = 2  # Binary classification: FRB vs RFI

## 3. Data Loading Functions

In [16]:
def load_frb_h5_files(frb_dir, target_size=TARGET_SIZE, use_freq_time=False):
    """
    Load FRB data from H5 files.
    
    Args:
        frb_dir: Directory containing FRB H5 files
        target_size: Target image dimensions (height, width)
        use_freq_time: If True, use freq-time data; if False, use DM-time data
    
    Returns:
        images: numpy array of shape (N, height, width, 1)
        labels: numpy array of shape (N,) with all 1s (FRB class)
        metadata: list of dictionaries with file info
    """
    h5_files = sorted(glob.glob(os.path.join(frb_dir, "*.h5")))
    
    if len(h5_files) == 0:
        raise FileNotFoundError(f"No H5 files found in {frb_dir}")
    
    print(f"Found {len(h5_files)} FRB H5 files")
    
    images = []
    metadata = []
    
    for h5_path in h5_files:
        try:
            with h5py.File(h5_path, 'r') as f:
                # Check what keys are available
                if len(metadata) == 0:  # First file - explore structure
                    print(f"\nExploring first file: {os.path.basename(h5_path)}")
                    print(f"Available keys: {list(f.keys())}")
                    
                # Try to load data - adapt to actual structure
                # Common patterns: 'data_freq_time', 'data_dm_time', 'freq_time', 'dm_time', 'data'
                data_key = None
                for key in ['data_freq_time', 'freq_time', 'data_dm_time', 'dm_time', 'data']:
                    if key in f:
                        data_key = key
                        break
                
                if data_key is None:
                    print(f"Warning: Could not find data in {os.path.basename(h5_path)}")
                    continue
                
                data = np.array(f[data_key])
                
                # Handle different data shapes
                if len(data.shape) == 2:
                    # (height, width) -> add channel dimension
                    data = data[..., np.newaxis]
                elif len(data.shape) == 4:
                    # (N, height, width, channels) -> extract first sample if batch
                    if data.shape[0] > 1:
                        print(f"Warning: Multiple samples in {os.path.basename(h5_path)}, using first")
                    data = data[0]
                
                # Resize to target size
                data_resized = resize(data, target_size).numpy()
                
                # Ensure channel dimension
                if len(data_resized.shape) == 2:
                    data_resized = data_resized[..., np.newaxis]
                
                images.append(data_resized)
                metadata.append({
                    'filename': os.path.basename(h5_path),
                    'original_shape': data.shape,
                    'data_key': data_key
                })
                
        except Exception as e:
            print(f"Error loading {os.path.basename(h5_path)}: {e}")
            continue
    
    if len(images) == 0:
        raise ValueError("No FRB images were successfully loaded!")
    
    images = np.array(images, dtype=np.float32)
    labels = np.ones(len(images), dtype=np.int32)  # All FRBs = class 1
    
    print(f"\nLoaded {len(images)} FRB images with shape {images.shape}")
    
    return images, labels, metadata

In [17]:
def load_rfi_pickle(pkl_path, target_size=TARGET_SIZE):
    """
    Load RFI data from pickle file.
    
    Args:
        pkl_path: Path to RFI pickle file
        target_size: Target image dimensions (height, width)
    
    Returns:
        images: numpy array of shape (N, height, width, 1)
        labels: numpy array of shape (N,) with all 0s (RFI class)
    """
    print(f"Loading RFI data from {pkl_path}...")
    
    with open(pkl_path, 'rb') as f:
        rfi_data = pickle.load(f)
    
    # Explore structure
    print(f"\nRFI pickle structure:")
    if isinstance(rfi_data, dict):
        print(f"  Keys: {list(rfi_data.keys())}")
        for key in rfi_data.keys():
            val = rfi_data[key]
            if isinstance(val, np.ndarray):
                print(f"  {key}: shape {val.shape}, dtype {val.dtype}")
            else:
                print(f"  {key}: type {type(val)}")
    elif isinstance(rfi_data, np.ndarray):
        print(f"  Direct array: shape {rfi_data.shape}, dtype {rfi_data.dtype}")
    else:
        print(f"  Type: {type(rfi_data)}")
    
    # Extract images - adapt to actual structure
    if isinstance(rfi_data, dict):
        # Try common keys
        for key in ['data', 'images', 'freq_time', 'dm_time', 'X']:
            if key in rfi_data:
                images_raw = rfi_data[key]
                break
        else:
            # Use first array-like value
            for val in rfi_data.values():
                if isinstance(val, np.ndarray):
                    images_raw = val
                    break
    else:
        images_raw = rfi_data
    
    # Ensure 4D: (N, height, width, channels)
    if len(images_raw.shape) == 3:
        # (N, height, width) -> add channel
        images_raw = images_raw[..., np.newaxis]
    elif len(images_raw.shape) == 2:
        # Single image (height, width) -> add batch and channel
        images_raw = images_raw[np.newaxis, ..., np.newaxis]
    
    # Resize all images
    print(f"Resizing RFI images from {images_raw.shape[1:3]} to {target_size}...")
    images = []
    for i in range(len(images_raw)):
        img_resized = resize(images_raw[i], target_size).numpy()
        images.append(img_resized)
    
    images = np.array(images, dtype=np.float32)
    labels = np.zeros(len(images), dtype=np.int32)  # All RFI = class 0
    
    print(f"Loaded {len(images)} RFI images with shape {images.shape}")
    
    return images, labels

## 4. Load and Prepare Dataset

In [18]:
# Load FRB data
print("="*60)
print("LOADING FRB DATA")
print("="*60)
frb_images, frb_labels, frb_metadata = load_frb_h5_files(FRB_H5_DIR)

print("\n" + "="*60)
print("LOADING RFI DATA")
print("="*60)
rfi_images, rfi_labels = load_rfi_pickle(RFI_PKL_PATH)

LOADING FRB DATA
Found 31 FRB H5 files

Exploring first file: FRB20181209A_22154974_beamformed.h5
Available keys: ['first_packet_recv_time', 'history', 'index_map', 'reverse_map', 'tiedbeam_baseband', 'tiedbeam_locations', 'tiedbeam_power', 'time0']

Exploring first file: FRB20181213A_22310619_beamformed.h5
Available keys: ['first_packet_recv_time', 'history', 'index_map', 'reverse_map', 'tiedbeam_baseband', 'tiedbeam_locations', 'tiedbeam_power', 'time0']

Exploring first file: FRB20181226E_23948798_beamformed.h5
Available keys: ['first_packet_recv_time', 'history', 'index_map', 'reverse_map', 'tiedbeam_baseband', 'tiedbeam_locations', 'tiedbeam_power', 'time0']

Exploring first file: FRB20181231A_24342210_beamformed.h5
Available keys: ['first_packet_recv_time', 'history', 'index_map', 'reverse_map', 'tiedbeam_baseband', 'tiedbeam_locations', 'tiedbeam_power', 'time0']

Exploring first file: FRB20181231C_24366209_beamformed.h5
Available keys: ['first_packet_recv_time', 'history', 'ind

ValueError: No FRB images were successfully loaded!

In [ ]:
# Combine datasets
X = np.concatenate([frb_images, rfi_images], axis=0)
y = np.concatenate([frb_labels, rfi_labels], axis=0)

print(f"\nCombined dataset:")
print(f"  Total samples: {len(X)}")
print(f"  FRB samples: {np.sum(y == 1)} ({100*np.sum(y==1)/len(y):.1f}%)")
print(f"  RFI samples: {np.sum(y == 0)} ({100*np.sum(y==0)/len(y):.1f}%)")
print(f"  Image shape: {X.shape[1:]}")
print(f"  Data range: [{X.min():.3f}, {X.max():.3f}]")

## 5. Data Quality Assessment

In [ ]:
# Visualize sample images from each class
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

# FRB samples
frb_indices = np.where(y == 1)[0]
rfi_indices = np.where(y == 0)[0]

for i in range(5):
    # FRBs
    idx = np.random.choice(frb_indices)
    axes[0, i].imshow(X[idx, :, :, 0], aspect='auto', cmap='viridis')
    axes[0, i].set_title(f'FRB Sample {i+1}')
    axes[0, i].axis('off')
    
    # RFI
    idx = np.random.choice(rfi_indices)
    axes[1, i].imshow(X[idx, :, :, 0], aspect='auto', cmap='viridis')
    axes[1, i].set_title(f'RFI Sample {i+1}')
    axes[1, i].axis('off')

plt.suptitle('Sample Images: FRB (top) vs RFI (bottom)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Check for data quality issues
print("\nDATA QUALITY CHECKS:")
print("="*60)

# Check for NaN or Inf values
nan_count = np.isnan(X).sum()
inf_count = np.isinf(X).sum()
print(f"NaN values: {nan_count}")
print(f"Inf values: {inf_count}")

if nan_count > 0 or inf_count > 0:
    print("  WARNING: Found invalid values! Replacing with 0...")
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

# Check for all-zero images
zero_images = []
for i in range(len(X)):
    if np.all(X[i] == 0):
        zero_images.append(i)

print(f"All-zero images: {len(zero_images)}")
if len(zero_images) > 0:
    print(f"  WARNING: Found {len(zero_images)} blank images!")
    print(f"  Indices: {zero_images[:10]}..." if len(zero_images) > 10 else f"  Indices: {zero_images}")

# Statistical summary
print(f"\nValue statistics per class:")
print(f"  FRB: mean={X[y==1].mean():.4f}, std={X[y==1].std():.4f}, range=[{X[y==1].min():.4f}, {X[y==1].max():.4f}]")
print(f"  RFI: mean={X[y==0].mean():.4f}, std={X[y==0].std():.4f}, range=[{X[y==0].min():.4f}, {X[y==0].max():.4f}]")

## 6. Data Normalization

In [ ]:
# Normalize to [0, 1] range for better training stability
X_min, X_max = X.min(), X.max()
if X_max > X_min:
    X_normalized = (X - X_min) / (X_max - X_min)
else:
    X_normalized = X
    print("WARNING: All values are the same - skipping normalization")

print(f"Normalized data range: [{X_normalized.min():.3f}, {X_normalized.max():.3f}]")

# Store normalization parameters for inference
normalization_params = {'min': X_min, 'max': X_max}
print(f"Normalization parameters: {normalization_params}")

## 7. Train/Validation/Test Split

In [ ]:
# Split into train, validation, and test sets
# 70% train, 15% validation, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X_normalized, y, test_size=0.3, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"\nDataset splits:")
print(f"  Training:   {len(X_train)} samples ({100*len(X_train)/len(X):.1f}%)")
print(f"    FRB: {np.sum(y_train==1)}, RFI: {np.sum(y_train==0)}")
print(f"  Validation: {len(X_val)} samples ({100*len(X_val)/len(X):.1f}%)")
print(f"    FRB: {np.sum(y_val==1)}, RFI: {np.sum(y_val==0)}")
print(f"  Test:       {len(X_test)} samples ({100*len(X_test)/len(X):.1f}%)")
print(f"    FRB: {np.sum(y_test==1)}, RFI: {np.sum(y_test==0)}")

# Convert labels to one-hot encoding
y_train_cat = to_categorical(y_train, num_classes=NUM_CLASSES)
y_val_cat = to_categorical(y_val, num_classes=NUM_CLASSES)
y_test_cat = to_categorical(y_test, num_classes=NUM_CLASSES)

## 8. Build FPGA-Optimized CNN Model

This architecture is optimized based on successful FPGA synthesis:
- **Key constraint**: Keep flatten layer output < 1000 neurons
- **Strategy**: Use MaxPooling2D layers to reduce spatial dimensions
- **Target**: 64x64 input → two Conv2D+MaxPool layers → ~450 neurons before dense layer

In [ ]:
def create_fpga_optimized_model(input_shape=(64, 64, 1), num_classes=2):
    """
    Create CNN model optimized for FPGA synthesis.
    
    Architecture:
    - Conv2D (2 filters) + MaxPool -> reduces 64x64 to 31x31
    - Conv2D (2 filters) + MaxPool -> reduces 31x31 to 15x15  
    - Flatten (450 neurons - well below 1000 limit)
    - Dense (num_classes) with softmax
    
    Total parameters: ~14K (highly efficient for FPGA)
    """
    model = models.Sequential([
        # Layer 1: Convolution + Pooling
        layers.Conv2D(
            filters=2,
            kernel_size=(3, 3),
            activation='relu',
            padding='valid',
            input_shape=input_shape,
            name='conv2d_1'
        ),
        layers.MaxPooling2D(pool_size=(2, 2), name='maxpool_1'),
        
        # Layer 2: Convolution + Pooling
        layers.Conv2D(
            filters=2,
            kernel_size=(3, 3),
            activation='relu',
            padding='valid',
            name='conv2d_2'
        ),
        layers.MaxPooling2D(pool_size=(2, 2), name='maxpool_2'),
        
        # Flatten and classify
        layers.Flatten(name='flatten'),
        layers.Dense(num_classes, activation='softmax', name='dense_output')
    ])
    
    return model

In [ ]:
# Create model
model = create_fpga_optimized_model(
    input_shape=(TARGET_SIZE[0], TARGET_SIZE[1], 1),
    num_classes=NUM_CLASSES
)

# Compile model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Display model architecture
model.summary()

# Calculate flatten layer size
flatten_layer = model.get_layer('flatten')
flatten_output_size = flatten_layer.output_shape[1]
print(f"\n{'='*60}")
print(f"FPGA SYNTHESIS COMPATIBILITY CHECK")
print(f"{'='*60}")
print(f"Flatten layer output size: {flatten_output_size} neurons")
print(f"Target limit: 1000 neurons")
print(f"Status: {'✓ PASS' if flatten_output_size < 1000 else '✗ FAIL - May have synthesis issues'}")
print(f"{'='*60}")

## 9. Train Model

In [ ]:
# Training configuration
EPOCHS = 50
BATCH_SIZE = 32

# Callbacks
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

# Train model
print("\nStarting training...")
print(f"Epochs: {EPOCHS}, Batch size: {BATCH_SIZE}")

history = model.fit(
    X_train, y_train_cat,
    validation_data=(X_val, y_val_cat),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

print("\nTraining complete!")

## 10. Training History Visualization

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy
axes[0].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.3)

# Loss
axes[1].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Print final metrics
final_train_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]
print(f"\nFinal Training Accuracy: {final_train_acc:.4f}")
print(f"Final Validation Accuracy: {final_val_acc:.4f}")

## 11. Model Evaluation on Test Set

In [ ]:
# Make predictions
y_pred_proba = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_proba, axis=1)
y_test_labels = np.argmax(y_test_cat, axis=1)

# Calculate metrics
test_accuracy = accuracy_score(y_test_labels, y_pred)
test_auc = roc_auc_score(y_test_labels, y_pred_proba[:, 1])

print("="*60)
print("TEST SET PERFORMANCE")
print("="*60)
print(f"Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"AUC-ROC:  {test_auc:.4f}")
print("="*60)

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test_labels, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues',
    xticklabels=['RFI', 'FRB'],
    yticklabels=['RFI', 'FRB'],
    cbar_kws={'label': 'Count'}
)
plt.xlabel('Predicted Label', fontsize=12, fontweight='bold')
plt.ylabel('True Label', fontsize=12, fontweight='bold')
plt.title('Confusion Matrix - Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Calculate additional metrics from confusion matrix
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn)  # True Positive Rate (Recall for FRB)
specificity = tn / (tn + fp)  # True Negative Rate
precision = tp / (tp + fp)    # Precision for FRB
f1_score = 2 * (precision * sensitivity) / (precision + sensitivity)

print("\nDetailed Metrics:")
print(f"  True Positives (FRB correctly identified):  {tp}")
print(f"  True Negatives (RFI correctly identified):  {tn}")
print(f"  False Positives (RFI misclassified as FRB): {fp}")
print(f"  False Negatives (FRB misclassified as RFI): {fn}")
print(f"\n  Sensitivity (FRB Detection Rate): {sensitivity:.4f}")
print(f"  Specificity (RFI Rejection Rate):  {specificity:.4f}")
print(f"  Precision (FRB Purity):            {precision:.4f}")
print(f"  F1-Score:                          {f1_score:.4f}")

In [ ]:
# Classification Report
print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)
print(classification_report(
    y_test_labels, 
    y_pred, 
    target_names=['RFI (Class 0)', 'FRB (Class 1)'],
    digits=4
))

## 12. ROC Curve Analysis

In [ ]:
# Compute ROC curve
fpr, tpr, thresholds = roc_curve(y_test_labels, y_pred_proba[:, 1])

# Plot ROC curve
plt.figure(figsize=(8, 8))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {test_auc:.4f})', linewidth=2.5, color='#2E86AB')
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=1.5)
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12, fontweight='bold')
plt.ylabel('True Positive Rate (Sensitivity)', fontsize=12, fontweight='bold')
plt.title('ROC Curve: FRB vs RFI Classification', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Find optimal threshold (Youden's J statistic)
j_scores = tpr - fpr
optimal_idx = np.argmax(j_scores)
optimal_threshold = thresholds[optimal_idx]
optimal_tpr = tpr[optimal_idx]
optimal_fpr = fpr[optimal_idx]

print(f"\nOptimal Classification Threshold: {optimal_threshold:.4f}")
print(f"  At this threshold:")
print(f"    True Positive Rate:  {optimal_tpr:.4f}")
print(f"    False Positive Rate: {optimal_fpr:.4f}")
print(f"    Youden's J:          {j_scores[optimal_idx]:.4f}")

## 13. Precision-Recall Curve

In [ ]:
# Compute Precision-Recall curve
precision_curve, recall_curve, pr_thresholds = precision_recall_curve(
    y_test_labels, y_pred_proba[:, 1]
)
avg_precision = average_precision_score(y_test_labels, y_pred_proba[:, 1])

# Plot Precision-Recall curve
plt.figure(figsize=(8, 8))
plt.plot(
    recall_curve, precision_curve, 
    label=f'PR Curve (AP = {avg_precision:.4f})', 
    linewidth=2.5, 
    color='#A23B72'
)
plt.xlabel('Recall (Sensitivity)', fontsize=12, fontweight='bold')
plt.ylabel('Precision', fontsize=12, fontweight='bold')
plt.title('Precision-Recall Curve: FRB Detection', fontsize=14, fontweight='bold')
plt.legend(loc='lower left', fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Average Precision Score: {avg_precision:.4f}")

## 14. Prediction Examples

In [ ]:
# Show some prediction examples
num_examples = 8
random_indices = np.random.choice(len(X_test), num_examples, replace=False)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()

for i, idx in enumerate(random_indices):
    true_label = y_test_labels[idx]
    pred_label = y_pred[idx]
    pred_proba = y_pred_proba[idx, 1]
    
    label_names = ['RFI', 'FRB']
    is_correct = true_label == pred_label
    
    axes[i].imshow(X_test[idx, :, :, 0], aspect='auto', cmap='viridis')
    
    title_color = 'green' if is_correct else 'red'
    title = f"True: {label_names[true_label]}\nPred: {label_names[pred_label]} ({pred_proba:.3f})"
    axes[i].set_title(title, color=title_color, fontweight='bold')
    axes[i].axis('off')

plt.suptitle('Prediction Examples (Green = Correct, Red = Incorrect)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 15. Save Model and Artifacts

In [ ]:
# Create output directory
output_dir = Path('model_outputs')
output_dir.mkdir(exist_ok=True)

# Save Keras model
model_path = output_dir / 'frb_classifier.keras'
model.save(model_path)
print(f"Saved Keras model to: {model_path}")

# Save model in HDF5 format (for hls4ml compatibility)
h5_model_path = output_dir / 'frb_classifier.h5'
model.save(h5_model_path)
print(f"Saved HDF5 model to: {h5_model_path}")

# Save normalization parameters
np.save(output_dir / 'normalization_params.npy', normalization_params)
print(f"Saved normalization parameters")

# Save training history
history_df = pd.DataFrame(history.history)
history_df.to_csv(output_dir / 'training_history.csv', index=False)
print(f"Saved training history")

# Save test set predictions
predictions_df = pd.DataFrame({
    'true_label': y_test_labels,
    'predicted_label': y_pred,
    'probability_rfi': y_pred_proba[:, 0],
    'probability_frb': y_pred_proba[:, 1]
})
predictions_df.to_csv(output_dir / 'test_predictions.csv', index=False)
print(f"Saved test predictions")

# Save model summary
with open(output_dir / 'model_summary.txt', 'w') as f:
    model.summary(print_fn=lambda x: f.write(x + '\n'))
print(f"Saved model summary")

# Save comprehensive metrics report
metrics_report = f"""
FRB vs RFI CLASSIFIER - PERFORMANCE REPORT
{'='*60}

MODEL ARCHITECTURE:
  Input Shape: {TARGET_SIZE} x 1
  Total Parameters: {model.count_params()}
  Trainable Parameters: {sum([tf.size(w).numpy() for w in model.trainable_weights])}
  Flatten Layer Size: {flatten_output_size} neurons

DATASET:
  Total Samples: {len(X)}
  Training: {len(X_train)} ({100*len(X_train)/len(X):.1f}%)
  Validation: {len(X_val)} ({100*len(X_val)/len(X):.1f}%)
  Test: {len(X_test)} ({100*len(X_test)/len(X):.1f}%)
  
TRAINING:
  Epochs Completed: {len(history.history['loss'])}
  Final Training Accuracy: {final_train_acc:.4f}
  Final Validation Accuracy: {final_val_acc:.4f}

TEST SET PERFORMANCE:
  Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)
  AUC-ROC: {test_auc:.4f}
  Average Precision: {avg_precision:.4f}
  
  Sensitivity (TPR): {sensitivity:.4f}
  Specificity (TNR): {specificity:.4f}
  Precision: {precision:.4f}
  F1-Score: {f1_score:.4f}

CONFUSION MATRIX:
  True Positives:  {tp}
  True Negatives:  {tn}
  False Positives: {fp}
  False Negatives: {fn}

OPTIMAL THRESHOLD:
  Threshold: {optimal_threshold:.4f}
  TPR at threshold: {optimal_tpr:.4f}
  FPR at threshold: {optimal_fpr:.4f}

FPGA DEPLOYMENT READINESS:
  Flatten neurons: {flatten_output_size} (< 1000: {'PASS' if flatten_output_size < 1000 else 'FAIL'})
  Model size: {model.count_params()} parameters
  Expected synthesis: {'Good' if flatten_output_size < 1000 else 'May have issues'}
  Target latency: <100 microseconds

{'='*60}
"""

with open(output_dir / 'performance_report.txt', 'w') as f:
    f.write(metrics_report)

print(f"\nSaved comprehensive performance report")
print(f"\nAll artifacts saved to: {output_dir.absolute()}")

## 16. Display Final Summary

In [ ]:
print(metrics_report)

## 17. Next Steps: FPGA Deployment

To deploy this model on FPGA using `hls4ml`, follow these steps:

```python
import hls4ml

# Load the saved model
model = keras.models.load_model('model_outputs/frb_classifier.h5')

# Configure hls4ml
config = hls4ml.utils.config_from_keras_model(
    model, 
    granularity='name',
    backend='Vivado'
)

# Set precision (ap_fixed<16,6> as per your successful tests)
config['Model']['Precision'] = 'ap_fixed<16,6>'
config['Model']['ReuseFactor'] = 1
config['Model']['Strategy'] = 'Resource'

# Convert to HLS
hls_model = hls4ml.converters.convert_from_keras_model(
    model,
    hls_config=config,
    output_dir='hls4ml_project',
    io_type='io_stream'
)

# Compile and synthesize
hls_model.compile()
hls_model.build(csim=True, synth=True, cosim=False, validation=False)
```

### Expected FPGA Performance:
- **Latency**: 50-100 microseconds (based on similar architecture)
- **DSP Usage**: ~8-10% of VU9P
- **LUT Usage**: ~2-3% of VU9P
- **Frequency**: ~250-300 MHz